In [4]:
import numpy as np
import pandas as pd

print('pandas:', pd.__version__, '| numpy:', np.__version__)

pandas: 2.2.3 | numpy: 2.4.6


In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load the SurvPath-filtered RNA-seq matrix uploaded to your Drive
DATA_PATH = '/content/drive/MyDrive/Datasets/fydp/rnaseq_filtered_survpath.csv'

df_rna = pd.read_csv(DATA_PATH, index_col=0)
print('Shape:', df_rna.shape)  # expected: (1073, ~4999)
df_rna.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (1073, 4906)


,A2M,AAAS,AADAT,AARS2,AASS,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,173.5472,32.6330,0.5864,26.3603,1.6383,12.8127,5.5212,82.3301,143.0760,0.1332,...,8.2165,1.7961,23.2207,13.0002,41.3931,0.0,0.1276,30.4353,82.3660,193.0483
TCGA-3C-AALI,193.2961,38.9292,1.8664,16.3475,2.2227,4.4627,6.9859,38.1435,60.4113,1.4034,...,9.9795,0.9161,12.7001,14.0755,46.5166,0.0,0.0000,21.3471,74.3577,254.0944
TCGA-3C-AALJ,204.7472,27.9897,4.0074,13.9400,2.1806,6.3691,10.7979,47.6396,31.7794,0.0810,...,3.4923,4.1575,14.1258,17.4336,20.7743,0.0,0.1054,17.4182,176.5328,225.4928
TCGA-3C-AALK,261.7484,30.4796,3.7265,13.4879,4.7480,28.4268,11.1177,20.3682,20.5225,1.7977,...,7.4476,4.2791,11.0752,8.7774,24.9127,0.0,0.0000,18.4196,47.9590,245.2738
TCGA-4H-AAAK,290.6471,41.2222,4.8974,23.2666,12.3309,17.5398,13.8918,17.5077,32.9598,1.3693,...,10.1026,4.0954,18.1065,17.6384,38.0379,0.0,0.0637,21.5292,62.4581,220.1794


In [4]:
%pip install -q pandas==2.2.3 anndata scanpy transformers huggingface_hub

In [1]:
from huggingface_hub import login
login()  # paste your Hugging Face token when prompted

In [2]:
from transformers import AutoModel, AutoTokenizer

model_name = "ScientaLab/eva-rna"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, device_map="auto")
model.eval()
print("Model loaded on:", next(model.parameters()).device)


Loading weights:   0%|          | 0/319 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [6]:
import anndata as ad
import scanpy as sc

# 1. Keep genes present in eva-rna's symbol_to_ncbi mapper
symbol_mapper = tokenizer.gene_mapper["symbol_to_ncbi"]
matched_genes = [g for g in df_rna.columns if g in symbol_mapper]
print(f"SurvPath genes: {df_rna.shape[1]} | Matched in eva-rna: {len(matched_genes)}")

df_rna_eva = df_rna[matched_genes]

# 2. Build AnnData and apply log2(TPM + 1)
adata = ad.AnnData(
    X=df_rna_eva.to_numpy(dtype="float32"),
    obs=pd.DataFrame(index=df_rna_eva.index),
    var=pd.DataFrame(index=df_rna_eva.columns),
)
sc.pp.log1p(adata, base=2)
print("AnnData ready:", adata.shape)
adata


SurvPath genes: 4906 | Matched in eva-rna: 4881
AnnData ready: (1073, 4881)


AnnData object with n_obs × n_vars = 1073 × 4881
    uns: 'log1p'

In [7]:
import os
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
model = model.half()

with torch.no_grad():
    embeddings = model.encode_anndata(
        tokenizer,
        adata,
        species="human",
        batch_size=4,
    )

embeddings = embeddings.astype("float32")
print("Embeddings shape:", embeddings.shape)

emb_cols = [f"eva_{i}" for i in range(embeddings.shape[1])]
df_eva_emb = pd.DataFrame(embeddings, index=adata.obs_names, columns=emb_cols)
df_eva_emb.to_parquet("rna_eva_survpath_embeddings.parquet")
df_eva_emb.to_parquet("/content/drive/MyDrive/rna_eva_survpath_embeddings.parquet")
print("Saved to Google Drive: /content/drive/MyDrive/rna_eva_survpath_embeddings.parquet")
df_eva_emb.head()


/root/.cache/huggingface/modules/transformers_modules/ScientaLab/eva_hyphen_rna/4232edf3c7efb01e4bb828be016f3fc86600f3e6/modeling_eva_rna.py:330: UserWarning: Auto-converted 4881/4881 genes from symbol to NCBI GeneIDs (species: human). Set species='human'/'mouse' explicitly if auto-detection is wrong.
  return encode_from_anndata(
Encoding samples:   2%|▏         | 5/269 [00:29<25:46,  5.86s/batch]


KeyboardInterrupt: 